# 01 - Análise Exploratória de Dados (EDA) & Baseline

Notebook para exploração inicial do dataset de telecomunicações e criação de um modelo baseline para previsão de Churn.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

%matplotlib inline

## 1. Carregamento dos Dados

Utilizamos a função `load_data` do módulo `src.data_loader` para carregar o dataset **Telco Customer Churn**.
Este dataset contém informações de clientes de uma empresa de telecomunicações, incluindo dados demográficos, serviços contratados e se o cliente cancelou (Churn = Yes) ou não (Churn = No).

In [ ]:
df = load_data('../data/raw/telco_churn.csv')
df.head()

In [ ]:
df.info()

## 2. Limpeza da coluna `TotalCharges`

Esta é uma **pegadinha clássica** deste dataset: a coluna `TotalCharges` é importada com dtype `object` (string) em vez de `float64`.

**Causa raiz:** existem registros com espaços em branco (`" "`) no lugar de valores numéricos. Esses registros pertencem a clientes novos com `tenure = 0`, ou seja, clientes que acabaram de entrar e ainda não tiveram nenhuma cobrança total registrada.

**Estratégia de tratamento:**
1. Converter a coluna para numérico com `pd.to_numeric(..., errors='coerce')` — valores inválidos viram `NaN`.
2. Remover as linhas com `NaN` usando `dropna()`. São apenas ~11 registros em ~7.000, um impacto desprezível.

In [ ]:
# Converte TotalCharges para numérico (espaços em branco viram NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"Valores nulos em TotalCharges: {df['TotalCharges'].isna().sum()}")
print(f"Total de linhas antes da limpeza: {len(df)}")

# Remove as linhas com valores nulos
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Total de linhas após limpeza: {len(df)}")
print(f"Nulos restantes: {df.isnull().sum().sum()}")

## 3. Distribuição da Variável Alvo (`Churn`)

Antes de qualquer modelagem, é fundamental entender o **balanceamento da variável alvo**.

Datasets de Churn são tipicamente **desbalanceados** — a maioria dos clientes permanece ativa e apenas uma fração cancela. Esse desbalanceamento impacta diretamente a escolha de métricas (acurácia pode ser enganosa) e pode exigir técnicas como oversampling, undersampling ou ajuste de pesos na função de perda da rede neural.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Churn', ax=ax, palette='Set2')
ax.set_title('Distribuição da Variável Alvo (Churn)')
ax.set_xlabel('Churn')
ax.set_ylabel('Quantidade de Clientes')

# Mostra as contagens e percentuais
total = len(df)
for p in ax.patches:
    count = int(p.get_height())
    pct = f'{100 * count / total:.1f}%'
    ax.annotate(f'{count}\n({pct})', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()